# 04 — Preparação do Dataset: Dados Públicos (RWF-2000)

> **Dados públicos são mantidos completamente separados dos dados do cliente.**

Gera o dataset YOLO em `data/datasets/public_only/` usando o RWF-2000  
(ou qualquer dataset público de violência).

Este dataset é usado:
- De forma **independente** no notebook 06 (treinamento combinado)
- Nunca misturado diretamente com `client_labeled/`

In [ ]:
!pip install -q ultralytics opencv-python-headless mediapipe scikit-learn tqdm PyYAML matplotlib

In [ ]:
import cv2, yaml, json
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

mp_pose = mp.solutions.pose

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PUBLIC_RAW_DIR = Path("../data/public_raw")           # coloque RWF-2000 aqui
DATASET_DIR    = Path("../data/datasets/public_only") # dataset YOLO público

EXTRACT_FPS  = 5
IMG_SIZE     = 640
PRE_V_THRESH = 0.45
CLASS_NAMES  = ["person", "violence", "pre_violence"]
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

for split in ["train","val","test"]:
    (DATASET_DIR/"images"/split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR/"labels"/split).mkdir(parents=True, exist_ok=True)
(DATASET_DIR/"metadata").mkdir(exist_ok=True)
print(f"Origem  : {PUBLIC_RAW_DIR.resolve()}")
print(f"Dataset : {DATASET_DIR.resolve()}")

## 1. Baixar RWF-2000 (opcional — via Kaggle)

In [ ]:
USE_KAGGLE = False   # ← True para baixar automaticamente

if USE_KAGGLE:
    !pip install -q kaggle
    !kaggle datasets download -d vulamnguyen/rwf2000 -p {PUBLIC_RAW_DIR}
    import zipfile
    with zipfile.ZipFile(PUBLIC_RAW_DIR/"rwf2000.zip") as z:
        z.extractall(PUBLIC_RAW_DIR)
    print("Download concluído.")

# Verificar estrutura esperada: public_raw/violence/ e public_raw/non_violence/
for label in ["violence","non_violence"]:
    d = PUBLIC_RAW_DIR / label
    n = len(list(d.glob("*.mp4")) + list(d.glob("*.avi"))) if d.exists() else 0
    print(f"  {label:15s}: {n} vídeos")

## 2. Processar e gerar dataset

In [ ]:
_yolo = YOLO("yolov8n.pt")

def detect_bboxes(frame):
    res = _yolo(frame, classes=[0], verbose=False)[0]
    boxes = [tuple(int(v) for v in b) for b in res.boxes.xyxy.tolist()]
    return boxes or [(0,0,frame.shape[1],frame.shape[0])]

def pose_score(frame):
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.4) as p:
        res = p.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if not res.pose_landmarks: return 0.0
        lm = res.pose_landmarks.landmark
        def pt(i): return np.array([lm[i].x, lm[i].y])
        try:
            shy  = (pt(11)[1]+pt(12)[1])/2
            arm  = min(1.0,(max(0,shy-pt(15)[1])+max(0,shy-pt(16)[1]))*2)
            tor  = min(1.0,abs(np.arctan2((pt(0)-(pt(23)+pt(24))/2)[0],
                                          -(pt(0)-(pt(23)+pt(24))/2)[1]))/(np.pi/3))
            spd  = min(1.0,np.linalg.norm(pt(15)-pt(16))*2)
            return float(np.clip(0.4*arm+0.3*tor+0.3*spd,0,1))
        except: return 0.0

def to_yolo(w,h,bbox,cls):
    x1,y1,x2,y2=bbox
    return f"{cls} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}"

def process_video(path, label):
    cap  = cv2.VideoCapture(str(path))
    if not cap.isOpened(): return []
    fps  = cap.get(cv2.CAP_PROP_FPS) or 25
    skip = max(1, int(fps/EXTRACT_FPS))
    recs, fi = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fi % skip == 0:
            h,w = frame.shape[:2]
            ps  = pose_score(frame)
            if label == "violence":    cls = 1
            elif ps >= PRE_V_THRESH:   cls = 2
            else:                      cls = 0
            bboxes = detect_bboxes(frame)
            recs.append(dict(frame=frame, class_id=cls,
                             annots=[to_yolo(w,h,b,cls) for b in bboxes]))
        fi += 1
    cap.release()
    return recs


all_recs = []
for label in ["violence","non_violence"]:
    vids = list((PUBLIC_RAW_DIR/label).glob("*.mp4")) + list((PUBLIC_RAW_DIR/label).glob("*.avi"))
    for v in tqdm(vids, desc=label):
        all_recs.extend(process_video(v, label))

print(f"Frames: {len(all_recs)}")
for i,n in enumerate(CLASS_NAMES):
    print(f"  {n}: {sum(1 for r in all_recs if r['class_id']==i)}")

In [ ]:
lbls  = [r["class_id"] for r in all_recs]
tr_i,tmp_i = train_test_split(range(len(all_recs)),test_size=0.3,stratify=lbls,random_state=42)
tmp_l = [lbls[i] for i in tmp_i]
vl_i,te_i  = train_test_split(tmp_i,test_size=0.33,stratify=tmp_l,random_state=42)

for split, idxs in [("train",tr_i),("val",vl_i),("test",te_i)]:
    for k,i in enumerate(tqdm(idxs, desc=split)):
        rec   = all_recs[i]
        frame = cv2.resize(rec["frame"],(IMG_SIZE,IMG_SIZE))
        name  = f"{split}_{k:06d}"
        cv2.imwrite(str(DATASET_DIR/"images"/split/f"{name}.jpg"), frame)
        (DATASET_DIR/"labels"/split/f"{name}.txt").write_text("\n".join(rec["annots"]))

ds_yaml = dict(path=str(DATASET_DIR.resolve()),
               train="images/train", val="images/val", test="images/test",
               nc=3, names=CLASS_NAMES)
with open(DATASET_DIR/"dataset.yaml","w") as f: yaml.dump(ds_yaml,f,default_flow_style=False)

stats = dict(source="public_only", total=len(all_recs),
             split=dict(train=len(tr_i),val=len(vl_i),test=len(te_i)),
             classes={n:sum(1 for r in all_recs if r["class_id"]==i)
                      for i,n in enumerate(CLASS_NAMES)})
with open(DATASET_DIR/"metadata"/"stats.json","w") as f: json.dump(stats,f,indent=2)

print(f"\n✓ Dataset public_only pronto")
print(f"  YAML: {DATASET_DIR/'dataset.yaml'}")
print("\nPróximos: 05_train_client_only.ipynb  e  06_train_combined.ipynb")